# Sesión 01 — De Excel/SQL a R: sintaxis y carga de datos
## Notebook espejo R

**Curso:** Fundamentos de programación en Python y R · **PEBIBA XVIII** · 2026
**Docente:** Cand. Dr. Jordan King Rodríguez Mallqui · UNI

> Este notebook es el **espejo R** del `S01_Sintaxis_y_Carga_PY.ipynb`. Mismas secciones, mismos conceptos, sintaxis distinta. Lee ambos en paralelo para construir el modelo mental bilingüe.

---

## Índice

1. Variables y tipos básicos
2. Vectores: tu primer contenedor
3. ⚠️ Trampa #1: Copy-on-modify (R) vs referencia (Python)
4. ⚠️ Trampa #2: Indexado base-1
5. Listas con nombre (≈ diccionarios de Python)
6. Carga de un CSV con readr
7. Inspección y diagnóstico inicial
8. Resumen y comparación con Python

## Caso de Negocio: Acme Corp B2B (mismo del notebook Python)

Mismo dataset, misma misión. Hoy aprendemos los fundamentos de R para que el siguiente sesión respondas la pregunta de tu jefe (top 5 productos Q4 por región) en CUALQUIERA de los dos lenguajes.

## Objetivos de Aprendizaje

Al final de este notebook podrás:

1. **Diferenciar** copy-on-modify (R) de asignación por referencia (Python) — Trampa #1.
2. **Cargar** un CSV en R con `readr::read_csv` o base R, manejando encoding y separador.
3. **Inspeccionar** un tibble con `glimpse()`, `summary()`, `head()`.
4. **Mapear** los conceptos de R con sus equivalentes en Python (notebook hermano).

## Imports y Configuración

In [3]:
# Tidyverse: ecosistema moderno de paquetes R para data science
library(tidyverse)   # incluye dplyr, ggplot2, readr, tidyr, purrr, etc.

# Configuración
options(scipen = 999)             # evita notación científica en outputs
options(tibble.print_max = 10)    # imprime máx 10 filas por defecto

# Constantes
DATASET_FILENAME <- 'acme_ventas.csv'
DATA_DIR <- '../data'

cat('R version:', R.version.string, '\n')
cat('Buscando dataset en:', normalizePath(DATA_DIR), '\n')

R version: R version 4.6.0 (2026-04-24 ucrt) 
Buscando dataset en: G:\Mi unidad\06Docencia\01Dictado de cursos\01PEBIBA\2026\Fundamentos_Python_R\03Material\01_Sesion_Sintaxis_Carga\data 


## 1. Variables y tipos básicos

**Diferencia con Python:** en R el operador idiomático de asignación es `<-` (no `=`, aunque también funciona).

*¿Por qué `<-`?* Razón histórica + flecha refleja "asigna a". `=` también funciona pero la convención R es `<-`.

In [4]:
# Variables
precio <- 100
iva <- 0.18
total <- precio * (1 + iva)

cat('Total inicial:', total, '\n')

# Cambio precio — total NO se actualiza (igual que Python)
precio <- 200
cat('Total después de cambiar precio:', total, '\n')

Total inicial: 118 
Total después de cambiar precio: 118 


In [5]:
# Tipos básicos
monto <- 1500.50
unidades <- 12L              # L sufijo = entero (sin L sería numeric/double)
moneda <- 'PEN'
es_b2b <- TRUE               # en R: TRUE/FALSE en mayúsculas (no True/False)

# typeof() es el equivalente a type() de Python
cat('monto:    ', monto, '   ←', typeof(monto), '\n')
cat('unidades: ', unidades, '       ←', typeof(unidades), '\n')
cat('moneda:   ', moneda, '      ←', typeof(moneda), '\n')
cat('es_b2b:   ', es_b2b, '       ←', typeof(es_b2b), '\n')

monto:     1500.5    ← double 
unidades:  12        ← integer 
moneda:    PEN       ← character 
es_b2b:    TRUE        ← logical 


### 💡 Pro-Tip: `paste()` y `glue()` para formateo

R no tiene f-strings nativas como Python. Las opciones son:
- `paste(...)` — concatena con espacio
- `paste0(...)` — concatena sin espacio
- `sprintf('%.2f', x)` — estilo C
- `glue::glue('Total: {total}')` — el más cercano a f-strings (requiere `library(glue)`)

## 2. Vectores: tu primer contenedor

**Diferencia clave con Python:** un vector R es **homogéneo** (todos los elementos del mismo tipo). En Python una lista puede mezclar tipos.

In [6]:
# Vector de regiones
regiones <- c('Lima', 'Arequipa', 'Cusco', 'Trujillo', 'Piura')

cat('Tengo', length(regiones), 'regiones\n')
cat('Primera región:', regiones[1], '\n')      # ← índice 1 (¡no 0!)
cat('Última región:', regiones[length(regiones)], '\n')

Tengo 5 regiones
Primera región: Lima 
Última región: Piura 


In [7]:
# ⚠️ ¿Qué pasa si mezclas tipos?
mezcla <- c('TX-000123', 1500.50, TRUE)
print(mezcla)
cat('Tipo del vector:', typeof(mezcla), '\n')   # "character" — R coerciona TODO a string

[1] "TX-000123" "1500.5"    "TRUE"     
Tipo del vector: character 


### ⚠️ Real-World Warning: coerción silenciosa

R no te da error cuando intentas mezclar tipos en un vector — silenciosamente convierte todo al tipo "más general" (carácter > numérico > lógico).

*En la vida real:* esto es fuente de bugs. Si esperabas números y obtuviste strings, tus operaciones aritméticas fallarán o darán NA. **Siempre verifica `typeof()`** después de construir vectores.

## 3. ⚠️ Trampa #1: Copy-on-modify (R) vs referencia (Python)

**Esta es la trampa más importante** — y se siente OPUESTA a Python.

En Python, `y = x` crea otro nombre al mismo objeto. En R, `y <- x` también… **hasta que modificas `y`**, momento en el cual R copia internamente.

In [8]:
# Predict: ¿qué imprime?
x <- c(1, 2, 3)
y <- x
y <- c(y, 4)            # equivalente a y.append(4) en Python

cat('x =', x, '\n')
cat('y =', y, '\n')

x = 1 2 3 
y = 1 2 3 4 


**Resultado:**
```
x = 1 2 3
y = 1 2 3 4
```

**¡Distinto a Python!** En R `x` quedó intacto. R aplicó **copy-on-modify**: cuando modificaste `y`, R duplicó silenciosamente la estructura.

*Modelo mental:*
```
Antes: x ──┐
 ├──> [1, 2, 3]
 y ──┘

Después de y <- c(y, 4):
 x ──> [1, 2, 3]
 y ──> [1, 2, 3, 4] ← R copió antes de modificar
```

### Comparación lado a lado con Python

| Operación | Python | R |
|---|---|---|
| Asignar | `y = x` | `y <- x` |
| Modificar | `y.append(4)` | `y <- c(y, 4)` |
| Resultado en `x` | `x` se modifica también | `x` queda intacto |
| Modelo | Referencia compartida | Copy-on-modify |

**Costo en memoria:** R puede consumir más RAM con datasets grandes porque copia. Por eso paquetes como `data.table` ofrecen modificación in-place explícita.

## 4. ⚠️ Trampa #2: Indexado base-1

**R empieza en 1.** Si vienes de Python (donde se empieza en 0), **es la fuente más común de off-by-one**.

In [9]:
vec <- c('a', 'b', 'c', 'd', 'e')

cat('vec[1] =', vec[1], '   ← primer elemento\n')
cat('vec[2] =', vec[2], '   ← segundo elemento\n')
cat('vec[5] =', vec[5], '   ← último (length=5)\n')

# ¿Qué pasa con vec[6]? — devuelve NA, no error
cat('vec[6] =', vec[6], '   ← FUERA de rango → NA (no error)\n')

# ¿Y con vec[-1]?
cat('vec[-1] =', vec[-1], '   ← OJO: NO es "último" como en Python; en R significa "todos MENOS el primero"\n')

vec[1] = a    ← primer elemento
vec[2] = b    ← segundo elemento
vec[5] = e    ← último (length=5)
vec[6] = NA    ← FUERA de rango → NA (no error)
vec[-1] = b c d e    ← OJO: NO es "último" como en Python; en R significa "todos MENOS el primero"


### ⚠️ Real-World Warning: índices negativos significan otra cosa en R

Si vienes de Python, esperabas que `vec[-1]` fuera el último elemento. **En R `vec[-1]` significa "todo menos el primer elemento"**.

Para obtener el último: `tail(vec, 1)` o `vec[length(vec)]`.

Esta es la trampa #2 más severa al portar código entre lenguajes.

In [10]:
# Slicing en R: 1:3 NO es exclusivo (incluye 3)
cat('vec[1:3] =', vec[1:3], '   ← elementos 1, 2 y 3 (3 INCLUIDO, distinto a Python)\n')
cat('vec[c(1,3,5)] =', vec[c(1,3,5)], '   ← elementos 1, 3 y 5 (selección no contigua)\n')

vec[1:3] = a b c    ← elementos 1, 2 y 3 (3 INCLUIDO, distinto a Python)
vec[c(1,3,5)] = a c e    ← elementos 1, 3 y 5 (selección no contigua)


## 5. Listas con nombre (≈ diccionarios de Python)

En R, una `list` con nombres es lo más cercano a un diccionario Python.

In [11]:
transaccion <- list(
  id = 'TX-000123',
  fecha = '2026-12-15',
  region = 'Lima',
  monto = 1500.50,
  es_b2b = TRUE
)

cat('ID:', transaccion$id, '\n')                # acceso con $
cat('Región:', transaccion[['region']], '\n')   # acceso con [[]]
cat('Claves:', names(transaccion), '\n')

ID: TX-000123 
Región: Lima 
Claves: id fecha region monto es_b2b 


### Comparación con diccionario Python

| Operación | Python | R |
|---|---|---|
| Crear | `{'k': v}` | `list(k = v)` |
| Acceder a clave | `d['k']` | `lst$k` o `lst[['k']]` |
| Listar claves | `d.keys()` | `names(lst)` |
| Pertenencia | `'k' in d` | `'k' %in% names(lst)` |

## 6. Carga de un CSV con readr 

In [12]:
# Forma básica con readr (parte de tidyverse)
df <- read_csv(file.path(DATA_DIR, DATASET_FILENAME))

cat('OK Dataset cargado:', nrow(df), 'filas x', ncol(df), 'columnas\n')

Rows: 5000 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (8): id_transaccion, quarter, region, producto, categoria, cliente_id, ...
dbl  (3): cantidad, precio_unit, descuento_pct
date (1): fecha

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


OK Dataset cargado: 5000 filas x 12 columnas


### ⚠️ Real-World Warning: Encoding y separadores en R

Si el CSV viene con encoding latin1 y separador `;`:

```r
# Con readr (tidyverse)
df <- read_delim(
 path,
 delim = ';',
 locale = locale(encoding = 'latin1')
)

# Con base R
df <- read.csv(
 path,
 sep = ';',
 fileEncoding = 'latin1'
)
```

*Diferencia clave con Python:* `readr` infiere tipos automáticamente y muestra qué hizo ("Parsed with column specification"). En pandas tienes que pedir `df.dtypes` explícitamente.

## 7. Inspección y diagnóstico inicial 

In [13]:
# 1) head — primeras filas (≈ Python: df.head())
head(df)

id_transaccion,fecha,quarter,region,producto,categoria,cantidad,precio_unit,descuento_pct,cliente_id,vendedor,notas
<chr>,<date>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
TX-000001,2026-02-02,Q1,Cusco,Monitor 27 4K,Tecnología,4,1839.73,0.05,CL-8570,Diego Salazar,Devolución parcial Q anterior
TX-000002,2026-10-10,Q4,Arequipa,Laptop Pro 15,Tecnología,25,4595.06,0.05,CL-5096,Lucía Romero,NA
TX-000003,2026-08-27,Q3,Trujillo,Lámpara LED,Mobiliario,28,92.18,0.20,CL-7254,Carlos Vega,Compra urgente
TX-000004,2026-06-10,Q2,Trujillo,Resma Papel A4,Suministros,36,26.70,0.00,CL-9665,Diego Salazar,"Pedido grande, descuento aplicado"
TX-000005,2026-06-08,Q2,Arequipa,Cartucho Tinta Negra,Suministros,29,73.53,0.00,CL-1432,Diego Salazar,Devolución parcial Q anterior
TX-000006,2026-11-10,Q4,Piura,Webcam HD,Electrónica,10,173.83,0.05,CL-1986,Ana Torres,Cliente premium


In [14]:
# 2) glimpse — vista panorámica con tipos (≈ Python: df.info())
glimpse(df)

Rows: 5,000
Columns: 12
$ id_transaccion <chr> "TX-000001", "TX-000002", "TX-000003", "TX-000004", "TX…
$ fecha          <date> 2026-02-02, 2026-10-10, 2026-08-27, 2026-06-10, 2026-0…
$ quarter        <chr> "Q1", "Q4", "Q3", "Q2", "Q2", "Q4", "Q1", "Q3", "Q1", "…
$ region         <chr> "Cusco", "Arequipa", "Trujillo", "Trujillo", "Arequipa"…
$ producto       <chr> "Monitor 27 4K", "Laptop Pro 15", "Lámpara LED", "Resma…
$ categoria      <chr> "Tecnología", "Tecnología", "Mobiliario", "Suministros"…
$ cantidad       <dbl> 4, 25, 28, 36, 29, 10, 18, 22, 49, 27, 32, 37, 18, 48, …
$ precio_unit    <dbl> 1839.73, 4595.06, 92.18, 26.70, 73.53, 173.83, 841.14, …
$ descuento_pct  <dbl> 0.05, 0.05, 0.20, 0.00, 0.00, 0.05, 0.15, 0.20, 0.10, 0…
$ cliente_id     <chr> "CL-8570", "CL-5096", "CL-7254", "CL-9665", "CL-1432", …
$ vendedor       <chr> "Diego Salazar", "Lucía Romero", "Carlos Vega", "Diego …
$ notas          <chr> "Devolución parcial Q anterior", NA, "Compra urgente", …


In [15]:
# 3) summary — estadística descriptiva (≈ Python: df.describe())
summary(df)

   id_transaccion     fecha                 quarter           region    
 Length   :5000   Min.   :2026-01-01   Length   :5000   Length   :5000  
 N.unique :5000   1st Qu.:2026-04-01   N.unique :   4   N.unique :   6  
 N.blank  :   0   Median :2026-06-30   N.blank  :   0   N.blank  :   0  
 Min.nchar:   9   Mean   :2026-07-01   Min.nchar:   2   Min.nchar:   4  
 Max.nchar:   9   3rd Qu.:2026-10-02   Max.nchar:   2   Max.nchar:   8  
                  Max.   :2026-12-31                                    
      producto        categoria       cantidad        precio_unit     
 Length   :5000   Length   :5000   Min.   :   1.00   Min.   :  26.61  
 N.unique :  12   N.unique :   4   1st Qu.:  12.00   1st Qu.: 114.14  
 N.blank  :   0   N.blank  :   0   Median :  25.00   Median : 340.89  
 Min.nchar:   9   Min.nchar:  10   Mean   :  34.79   Mean   :1066.89  
 Max.nchar:  20   Max.nchar:  11   3rd Qu.:  37.00   3rd Qu.:1721.15  
                                   Max.   :9999.00   Max.   :47

In [16]:
# Helpers extra

# Conteo de nulos por columna
cat('Nulos por columna:\n')
df |> summarise(across(everything(), ~ sum(is.na(.x)))) |> glimpse()

# Filas duplicadas
cat('\nFilas duplicadas:', sum(duplicated(df)), '\n')

# Valores únicos en columnas categóricas
cat('\nValores únicos:\n')
for (col in c('quarter', 'region', 'categoria')) {
  cat(' ', col, ':', paste(unique(df[[col]]), collapse=', '), '\n')
}

Nulos por columna:
Rows: 1
Columns: 12
$ id_transaccion <int> 0
$ fecha          <int> 0
$ quarter        <int> 0
$ region         <int> 0
$ producto       <int> 0
$ categoria      <int> 0
$ cantidad       <int> 0
$ precio_unit    <int> 0
$ descuento_pct  <int> 0
$ cliente_id     <int> 0
$ vendedor       <int> 0
$ notas          <int> 1473

Filas duplicadas: 0 

Valores únicos:
  quarter : Q1, Q4, Q3, Q2 
  region : Cusco, Arequipa, Trujillo, Piura, Lima, Chiclayo 
  categoria : Tecnología, Mobiliario, Suministros, Electrónica 


## 8. Resumen y comparación con Python

**Hoy aprendiste R:**

1. ✅ Sintaxis básica con asignación `<-` y tipos.
2. ✅ Vectores homogéneos (¡no como listas Python!).
3. ✅ Trampa #1 inversa: copy-on-modify (R) vs referencia (Python).
4. ✅ Trampa #2 inversa: indexado base-1 (R) vs base-0 (Python).
5. ✅ Listas con nombre como equivalente de diccionarios Python.
6. ✅ Carga de CSV con `readr::read_csv`.
7. ✅ Inspección con `head()`, `glimpse()`, `summary()`.

### Tabla resumen Python ↔ R

| Operación | Python | R |
|---|---|---|
| Asignación | `x = 5` | `x <- 5` |
| Crear secuencia | `[1, 2, 3]` | `c(1, 2, 3)` |
| Largo | `len(x)` | `length(x)` |
| Primer elemento | `x[0]` | `x[1]` |
| Último elemento | `x[-1]` | `tail(x, 1)` |
| Mutación | Por referencia | Copy-on-modify |
| Diccionario | `{'k': v}` | `list(k = v)` |
| Cargar CSV | `pd.read_csv(p)` | `read_csv(p)` |
| Vista | `df.head()` | `head(df)` |
| Tipos+nulos | `df.info()` | `glimpse(df)` |
| Estadística | `df.describe()` | `summary(df)` |

 Esta tabla está expandida en el archivo **`S01_Sintaxis_y_Carga_ROSETTA.md`** — consúltala todo el curso.

## Siguiente Paso

**Sesión 02:** manipulación con `dplyr` (filter, group_by, summarise) — el equivalente de pandas. Veremos lado a lado y elegirás cuál se siente más natural para tu trabajo.

**Tarea pre-S02:**

1. Lee `S01_Sintaxis_y_Carga_TRAPS.md` — las 5 trampas asimétricas que vimos hoy.
2. Familiarízate con `S01_Sintaxis_y_Carga_ROSETTA.md`.

---

*Cierre Sesión 01 (espejo R) — Fundamentos de Python y R · PEBIBA XVIII · 2026*

---

## Profundización R (opcional)

> Esta sección es **opcional** — solo si ya terminaste todo lo demás y quieres explorar territorio que Python no resuelve igual de elegante.

Aquí verás **3 cosas** que R hace mejor (o distinto) que Python:

1. **Factors** — categorías ordenadas con niveles fijos
2. **data.table** — alternativa rápida a dplyr para datasets grandes
3. **purrr::map** — patrones funcionales más expresivos que las list comprehensions


### A. Factors: categorías con orden semántico

En el dataset Acme Corp, la columna `quarter` tiene valores `Q1`, `Q2`, `Q3`, `Q4`. R los ordena alfabéticamente por defecto, pero **semánticamente** queremos `Q1 < Q2 < Q3 < Q4`.

Los **factors ordenados** lo resuelven en una línea.

In [17]:
# Sin factor: orden alfabetico (funciona por casualidad con Q1..Q4)
df_demo <- data.frame(quarter = c('Q3', 'Q1', 'Q4', 'Q2'),
                       ventas  = c(100, 80, 120, 95))

# Con factor ordenado: orden semantico explicito
df_demo$quarter <- factor(df_demo$quarter, levels = c('Q1', 'Q2', 'Q3', 'Q4'), ordered = TRUE)
print(df_demo[order(df_demo$quarter), ])

# Bonus: forcats hace esto mas idiomatico
library(forcats)
df_demo$quarter <- fct_reorder(df_demo$quarter, df_demo$ventas)
print(levels(df_demo$quarter))

  quarter ventas
2      Q1     80
4      Q2     95
1      Q3    100
3      Q4    120
[1] "Q1" "Q2" "Q3" "Q4"


**¿Por qué importa?** `ggplot2` respeta el orden del factor automáticamente. Si pasas `quarter` como factor ordenado, las barras del gráfico salen Q1→Q4 sin que tengas que hacer nada. En pandas necesitas `pd.Categorical(..., ordered=True, categories=[...])` que es más verboso.


### B. data.table: rapidez en datasets grandes

`dplyr` es legible. `data.table` es **rápido** (a costa de sintaxis menos amigable). Heurística: < 1M filas usa dplyr. Más de 10M filas considera data.table.

Ambos resuelven la misma pregunta — Top 5 productos por ingreso total.

In [18]:
library(dplyr)
library(data.table)

# dplyr (legible)
top_dplyr <- df |>
  mutate(ingreso = precio_unit * cantidad) |>
  group_by(producto) |>
  summarise(total = sum(ingreso)) |>
  arrange(desc(total)) |>
  head(5)

# data.table (rapido — sintaxis tipo SQL en una expresion)
dt <- as.data.table(df)
top_dt <- dt[, .(total = sum(precio_unit * cantidad)), by = producto][order(-total)][1:5]

print(top_dplyr)
print(top_dt)


Adjuntando el paquete: 'data.table'


The following objects are masked from 'package:lubridate':

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from 'package:dplyr':

    between, first, last


The following object is masked from 'package:purrr':

    transpose


The following object is masked from 'package:base':

    %notin%




# A tibble: 5 × 2
  producto                 total
  <chr>                    <dbl>
1 Laptop Air 13        65468708.
2 Laptop Pro 15        47494216.
3 Escritorio Ajustable 24885183.
4 Monitor 27 4K        19101613.
5 Silla Ergonómica      8875666.
               producto    total
                 <char>    <num>
1:        Laptop Air 13 65468708
2:        Laptop Pro 15 47494216
3: Escritorio Ajustable 24885183
4:        Monitor 27 4K 19101613
5:     Silla Ergonómica  8875666


**Lectura del data.table:** `dt[i, j, by]` — `i` filtra filas, `j` calcula columnas, `by` agrupa. Es SQL en 3 letras.

*En tu trabajo:* si te dan datasets de >5M filas y `dplyr` se siente lento, vuelve aquí.


### C. purrr::map — aplicar una función a cada elemento

`purrr` es el **functional programming** de R, equivalente a las list comprehensions de Python pero más expresivo.

Caso de uso real: leer 12 archivos CSV mensuales y consolidarlos.

In [19]:
library(purrr)
library(dplyr)

# Dividir el dataset por region (lista de 6 dataframes)
por_region <- df |> group_by(region) |> group_split()

# Aplicar la misma agregacion a cada region
resumen_por_region <- map(por_region, function(df_r) {
  df_r |>
    summarise(region     = first(region),
              n_filas    = n(),
              ventas_tot = sum(precio_unit * cantidad),
              prod_unico = n_distinct(producto))
})

# Aplanar a un solo data.frame
resumen <- bind_rows(resumen_por_region)
print(resumen)

# A tibble: 6 × 4
  region   n_filas ventas_tot prod_unico
  <chr>      <int>      <dbl>      <int>
1 Arequipa     811  19380345.         12
2 Chiclayo     817  24417612.         12
3 Cusco        834  27588873.         12
4 Lima         860  35201865.         12
5 Piura        796  21783758.         12
6 Trujillo     882  53121644.         12


**Equivalente en Python:** `df.groupby('region').agg(...)` — pandas lo resuelve en una línea sin explicitar el `map`.

**Por qué purrr brilla:** cuando los datasets están en **archivos separados** (no una sola tabla), `map_dfr` aplica una función a cada archivo y consolida.

```r
# Leer 12 CSVs mensuales en 3 lineas:
library(readr)
todos <- list.files('data/mensuales/', '*.csv', full.names = TRUE) |>
 map_dfr(read_csv, .id = 'archivo_origen')
```

*Equivalente pandas:* ~7 líneas con `glob` + `pd.concat`.

### Cierre de la profundización

| Tema | Cuándo lo vas a usar |
|---|---|
| Factors ordenados | Visualización con orden semántico |
| data.table | Velocidad en datasets > 5M filas o joins pesados |
| purrr::map | Procesar muchos archivos similares (ETL mensual/diario) |

*Si quieres profundizar: Wickham — Advanced R (en `recursos/bibliografia.md`).*